<a href="https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [20]:
%pip install -q -U huggingface_hub duckdb

In [1]:
import os
from google.colab import userdata
from huggingface_hub import whoami

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN nahi mila. Colab ke Secrets panel mein HF_TOKEN add karo."
    )

# Check token validity without printing the token
user_info = whoami(token=HF_TOKEN)
print("Hugging Face login successful:", user_info["name"])

Hugging Face login successful: zafar4050


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [7]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [13]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print("Total clients:", len(clients))
clients.head(10)

Total clients: 104


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
7,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [9]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [10]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159,15.0,0.144623,0.665019,79.0,308.0,0.256494
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091,101.0,0.037423,0.178737,15557.0,18432.0,0.844021
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206,3.0,0.215054,0.623656,25.0,60.0,0.416667
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655,16.0,0.032740,0.717915,473.0,952.0,0.496849
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483,8.0,0.224066,0.630705,30.0,140.0,0.214286


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [11]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.548     0.330     0.412      9389
           1      0.684     0.842     0.755     16162

    accuracy                          0.654     25551
   macro avg      0.616     0.586     0.583     25551
weighted avg      0.634     0.654     0.629     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [14]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, accuracy_score

# =========================================================
# 1. Build a 90-day feature table
#    Previous 60 days = features
#    Last 30 days = outcome window
# =========================================================

features_90 = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            -- Outcome window: last 30 days
            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN COALESCE(f.gsc_impressions, 0)
                    ELSE 0
                END
            ) AS imp_last30,

            -- Feature window: previous 60 days
            SUM(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                     AND f.report_date > b.end_d - INTERVAL 90 DAY
                    THEN COALESCE(f.gsc_impressions, 0)
                    ELSE 0
                END
            ) AS imp_prev60,

            SUM(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                     AND f.report_date > b.end_d - INTERVAL 90 DAY
                    THEN COALESCE(f.gsc_clicks, 0)
                    ELSE 0
                END
            ) AS clicks_prev60,

            AVG(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                     AND f.report_date > b.end_d - INTERVAL 90 DAY
                    THEN f.gsc_avg_position
                END
            ) AS avg_position_prev60,

            -- New feature: position volatility
            STDDEV_SAMP(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                     AND f.report_date > b.end_d - INTERVAL 90 DAY
                    THEN f.gsc_avg_position
                END
            ) AS position_volatility,

            COUNT(
                DISTINCT CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                     AND f.report_date > b.end_d - INTERVAL 90 DAY
                     AND f.gsc_impressions > 0
                    THEN f.report_date
                END
            ) AS active_days_prev60

        FROM {TABLES['fact_daily']} f
        CROSS JOIN bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY
            f.client_hash_id,
            f.content_hash_id

        HAVING imp_prev60 >= 200
    )

    SELECT *
    FROM windowed
""").df()

print(f"{len(features_90):,} content items in the 90-day experiment")
features_90.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

114,435 content items in the 90-day experiment


,client_hash_id,content_hash_id,imp_last30,imp_prev60,clicks_prev60,avg_position_prev60,position_volatility,active_days_prev60
0,client_e547b89c05043229,content_d0fa1bbfbc10caf8,1628.0,4368.0,8.0,19.436140,8.006934,60
1,client_e547b89c05043229,content_4c1e972bec56132e,7098.0,19584.0,176.0,9.334575,4.840279,60
2,client_e547b89c05043229,content_64cad58fc02e7605,290.0,849.0,0.0,58.033683,19.149336,58
3,client_e547b89c05043229,content_4e48bd81bb37eb4f,23065.0,42460.0,24.0,34.333887,6.044934,60
4,client_e547b89c05043229,content_f338440914b1ab00,1460.0,4521.0,7.0,20.035685,7.159876,60


In [15]:
experiment_data = features_90.merge(
    qsignals[
        [
            "content_hash_id",
            "visible_queries",
            "rare_share",
            "anon_share",
            "top_query_share"
        ]
    ],
    on="content_hash_id",
    how="left"
)

print(f"Joined experiment data: {len(experiment_data):,} rows")
experiment_data.head()

Joined experiment data: 114,435 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev60,clicks_prev60,avg_position_prev60,position_volatility,active_days_prev60,visible_queries,rare_share,anon_share,top_query_share
0,client_e547b89c05043229,content_d0fa1bbfbc10caf8,1628.0,4368.0,8.0,19.436140,8.006934,60,20.0,0.052035,0.822882,0.152000
1,client_e547b89c05043229,content_4c1e972bec56132e,7098.0,19584.0,176.0,9.334575,4.840279,60,85.0,0.095233,0.734240,0.190549
2,client_e547b89c05043229,content_64cad58fc02e7605,290.0,849.0,0.0,58.033683,19.149336,58,6.0,0.166813,0.736611,0.400000
3,client_e547b89c05043229,content_4e48bd81bb37eb4f,23065.0,42460.0,24.0,34.333887,6.044934,60,340.0,0.035162,0.351316,0.228974
4,client_e547b89c05043229,content_f338440914b1ab00,1460.0,4521.0,7.0,20.035685,7.159876,60,30.0,0.099983,0.707741,0.185217


In [16]:
# Average 30-day equivalent impressions from the previous 60 days
experiment_data["expected_30_from_prev60"] = (
    experiment_data["imp_prev60"] / 2
)

# Positive label: last 30 days are more than 20% below
# the previous 60-day monthly-equivalent level
experiment_data["is_declining"] = (
    experiment_data["imp_last30"]
    < 0.8 * experiment_data["expected_30_from_prev60"]
).astype(int)

print(
    "Declining rate:",
    round(experiment_data["is_declining"].mean(), 3)
)

Declining rate: 0.686


In [17]:
feature_cols_exp = [
    "imp_prev60",
    "clicks_prev60",
    "avg_position_prev60",
    "position_volatility",
    "active_days_prev60",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share",
]

model_data_exp = experiment_data.dropna(
    subset=feature_cols_exp + ["client_hash_id", "is_declining"]
).copy()

X_exp = model_data_exp[feature_cols_exp]
y_exp = model_data_exp["is_declining"]
groups = model_data_exp["client_hash_id"]

group_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    group_split.split(X_exp, y_exp, groups=groups)
)

X_train = X_exp.iloc[train_idx]
X_test = X_exp.iloc[test_idx]

y_train = y_exp.iloc[train_idx]
y_test = y_exp.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print(
    "Client overlap:",
    len(train_clients.intersection(test_clients))
)

Train rows: 92173
Test rows: 12435
Train clients: 38
Test clients: 13
Client overlap: 0


In [18]:
group_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

group_model.fit(X_train, y_train)

group_predictions = group_model.predict(X_test)

base_rate = max(
    y_test.mean(),
    1 - y_test.mean()
)

group_accuracy = accuracy_score(
    y_test,
    group_predictions
)

print(f"Majority-class baseline: {base_rate:.3f}")
print(f"Group-split model accuracy: {group_accuracy:.3f}")
print()
print(
    classification_report(
        y_test,
        group_predictions,
        digits=3,
        zero_division=0
    )
)

Majority-class baseline: 0.741
Group-split model accuracy: 0.839

              precision    recall  f1-score   support

           0      0.704     0.649     0.676      3215
           1      0.881     0.905     0.893      9220

    accuracy                          0.839     12435
   macro avg      0.793     0.777     0.784     12435
weighted avg      0.835     0.839     0.837     12435



In [19]:
feature_importance = (
    pd.DataFrame({
        "feature": feature_cols_exp,
        "importance": group_model.feature_importances_
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance

,feature,importance
0,imp_prev60,0.137396
1,rare_share,0.129413
2,anon_share,0.128157
3,avg_position_prev60,0.121434
4,position_volatility,0.116591
5,active_days_prev60,0.111487
6,top_query_share,0.101022
7,visible_queries,0.086850
8,clicks_prev60,0.067648


I extended the feature window from 30 days to 90 days and added additional search performance features such as average position, clicks, active days, and position volatility. I also filtered out low-volume pages using a minimum impression threshold.

The model was evaluated using GroupShuffleSplit, ensuring that pages from the same client never appeared in both the training and testing sets.

The model achieved an accuracy of 0.839, which is substantially higher than the majority-class baseline of 0.741. The most important feature was previous impressions (imp_prev60), followed by rare query share, anonymized query share, average position, and position volatility. These results suggest that historical search visibility and ranking stability carry useful predictive information for identifying declining pages.